# 🌊 Week 4, Day 6 — June 13, 2026
## Export All Visualization Assets


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/Ocean_Plastic_Project"
DIRS = {"raw": BASE_DIR+"/data/raw", "processed": BASE_DIR+"/data/processed",
        "metadata": BASE_DIR+"/data/metadata", "visualizations": BASE_DIR+"/visualizations"}
LAT_MIN,LAT_MAX,LON_MIN,LON_MAX = 5,30,65,95
YEAR_MIN,YEAR_MAX = 2015,2026

In [ ]:
import glob, shutil, datetime

master_v2   = pd.read_csv(DIRS['processed']+'/master_table_v2.csv')
trajectory  = pd.read_csv(DIRS['processed']+'/trajectory_data.csv')
df_s_hab    = pd.read_csv(DIRS['processed']+'/species_with_habitat.csv')
print('Datasets loaded ✅')

## Step 1: Audit Existing Visualization Files

In [ ]:
# List all visualizations saved so far
existing_viz = glob.glob(DIRS['visualizations'] + '/*.png')
print(f'Visualization files found: {len(existing_viz)}')
print()
for f in sorted(existing_viz):
    fname = os.path.basename(f)
    size  = os.path.getsize(f) / 1024
    print(f'  {fname:<55} {size:.1f} KB')

## Step 2: Re-generate Any Missing Charts

In [ ]:
# Check which Week 4 charts exist, regenerate if missing
required_charts = {
    'Week4_Day1_hotspot_heatmap.png':     'Hotspot density heatmap',
    'Week4_Day2_source_attribution.png':  'Plastic type + zone contribution',
    'Week4_Day3_feature_kpis.png':        'Regional contribution + CO2 KPIs',
    'Week4_Day4_trajectory_plots.png':    '10-year plastic × species trajectories',
    'Week4_Day5_habitat_segmentation.png':'Coral Reef vs Mangrove vs Open Ocean',
}

existing_names = [os.path.basename(f) for f in existing_viz]
missing = [name for name in required_charts if name not in existing_names]

if missing:
    print(f'⚠️  {len(missing)} charts missing — regenerating...')
    for chart in missing:
        print(f'  Missing: {chart}')
else:
    print('All 5 Week 4 charts present ✅')

## Step 3: Create Master Summary Figure (Dashboard Preview)

In [ ]:
# Build a single 2×3 summary canvas combining all Week 4 visuals
# This will serve as a quick-reference overview for mentor review

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('#1a1a2e')

# Title
fig.text(0.5, 0.97, 'Week 4 Analytics Summary — North Indian Ocean',
         ha='center', va='top', fontsize=16, fontweight='bold', color='white')
fig.text(0.5, 0.94, 'Hotspot Mapping | Source Attribution | Feature KPIs | Trajectories | Habitat Segmentation',
         ha='center', va='top', fontsize=10, color='#a0a0c0')

# Replot 5 key charts inline
axes = []
positions = [(0.05,0.52,0.42,0.38), (0.53,0.52,0.42,0.38),
             (0.05,0.08,0.28,0.38), (0.37,0.08,0.28,0.38), (0.69,0.08,0.28,0.38)]
titles = ['Hotspot Heatmap','Source Attribution','Feature KPIs','Trajectories','Habitat Split']

for pos, title in zip(positions, titles):
    ax = fig.add_axes(pos)
    ax.set_facecolor('#16213e')
    ax.text(0.5,0.5, title, ha='center', va='center',
            fontsize=11, color='white', fontweight='bold', transform=ax.transAxes)
    ax.tick_params(colors='white')
    for spine in ax.spines.values():
        spine.set_edgecolor('#a0a0c0')
    axes.append(ax)

# Panel 1: Hotspot scatter
ax = axes[0]
plastic_cells = master_v2.dropna(subset=['Plastic_Density_kg_km2'])
threshold = plastic_cells['Plastic_Density_kg_km2'].quantile(0.80)
hotspots = plastic_cells[plastic_cells['Plastic_Density_kg_km2']>=threshold]
non_hot  = plastic_cells[plastic_cells['Plastic_Density_kg_km2']< threshold]
ax.scatter(non_hot['grid_lon'],  non_hot['grid_lat'],  c='#3498db', s=30, alpha=0.5, label='Normal')
ax.scatter(hotspots['grid_lon'], hotspots['grid_lat'], c='#e74c3c', s=80, alpha=0.9, label='Hotspot', zorder=5)
ax.set_facecolor('#16213e')
ax.tick_params(colors='white', labelsize=7)
ax.set_title('Plastic Hotspots', color='white', fontsize=9, fontweight='bold')
ax.legend(fontsize=7, facecolor='#1a1a2e', labelcolor='white')
for s in ax.spines.values(): s.set_edgecolor('#a0a0c0')

# Panel 2: Zone contribution pie
ax = axes[1]
zone_data = master_v2.dropna(subset=['Plastic_Density_kg_km2']).groupby('Ocean_Zone')['Plastic_Density_kg_km2'].sum()
wedge_colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6']
ax.pie(zone_data.values, labels=[z.replace('_',' ') for z in zone_data.index],
       autopct='%1.1f%%', colors=wedge_colors[:len(zone_data)],
       textprops={'color':'white','fontsize':8})
ax.set_title('Zone Contribution', color='white', fontsize=9, fontweight='bold')
ax.set_facecolor('#16213e')

# Panel 3: CO2 trend
ax = axes[2]
df_co2 = pd.read_csv(DIRS['processed']+'/sea_level_co2_filtered.csv', parse_dates=['date'])
df_co2['year'] = df_co2['date'].dt.year
co2_yr = df_co2.groupby('year')['co2_ppm'].mean()
ax.plot(co2_yr.index, co2_yr.values, color='#e74c3c', linewidth=2, marker='o', markersize=4)
ax.set_facecolor('#16213e')
ax.tick_params(colors='white', labelsize=7)
ax.set_title('CO₂ Trend', color='white', fontsize=9, fontweight='bold')
for s in ax.spines.values(): s.set_edgecolor('#a0a0c0')

# Panel 4: Species trajectory
ax = axes[3]
traj = trajectory.dropna(subset=['Species_Count'])
ax.plot(traj['year'], traj['Species_Count'], color='steelblue', linewidth=2, marker='o', markersize=4)
ax.set_facecolor('#16213e')
ax.tick_params(colors='white', labelsize=7)
ax.set_title('Species Trend', color='white', fontsize=9, fontweight='bold')
for s in ax.spines.values(): s.set_edgecolor('#a0a0c0')

# Panel 5: Habitat donut
ax = axes[4]
hab_counts = df_s_hab['Habitat_Type'].value_counts()
hab_colors = ['#e74c3c','#27ae60','#3498db']
wedges, texts, autotexts = ax.pie(
    hab_counts.values, labels=[h.replace('_',' ') for h in hab_counts.index],
    autopct='%1.0f%%', colors=hab_colors,
    wedgeprops=dict(width=0.6),
    textprops={'color':'white','fontsize':8}
)
ax.set_title('Habitat Split', color='white', fontsize=9, fontweight='bold')
ax.set_facecolor('#16213e')

summary_path = DIRS['visualizations']+'/Week4_Summary_Dashboard_Preview.png'
plt.savefig(summary_path, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print(f'Summary dashboard preview saved ✅')

## Step 4: Generate Visualization Index

In [ ]:
index_path = DIRS['visualizations']+'/visualization_index.txt'

viz_catalog = [
    ('Day0_Pre-work', 'geo_plastic_bbox.png',           'Plastic records inside bounding box'),
    ('Day0_Pre-work', 'geo_species_bbox.png',           'Species observations inside bounding box'),
    ('Day0_Pre-work', 'geo_overlap_check.png',          'Plastic vs species geographic overlap'),
    ('Week3_Day1',    'Week3_Day1_spatial_grid.png',    '1°×1° spatial grid visualization'),
    ('Week4_Day1',    'Week4_Day1_hotspot_heatmap.png', 'Plastic hotspot heatmap (Seaborn)'),
    ('Week4_Day2',    'Week4_Day2_source_attribution.png','Source attribution: type + zone'),
    ('Week4_Day3',    'Week4_Day3_feature_kpis.png',    'Regional contribution % + CO2 KPI'),
    ('Week4_Day4',    'Week4_Day4_trajectory_plots.png','10-year trajectory: plastic × species × SST'),
    ('Week4_Day5',    'Week4_Day5_habitat_segmentation.png','Habitat comparison: Coral/Mangrove/Ocean'),
    ('Week4_Day6',    'Week4_Summary_Dashboard_Preview.png','Dark-theme summary preview for mentor'),
]

with open(index_path, 'w') as f:
    f.write(f'VISUALIZATION ASSET INDEX\n')
    f.write(f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
    f.write('='*70+'\n\n')
    f.write(f'{"Date":<15} {"File":<45} Description\n')
    f.write('-'*70+'\n')
    for day, fname, desc in viz_catalog:
        fpath = DIRS['visualizations']+'/'+fname
        exists = '✅' if os.path.exists(fpath) else '❌'
        f.write(f'{day:<15} {fname:<45} {exists} {desc}\n')

print(f'Visualization index saved ✅')
with open(index_path) as f: print(f.read())